# `lg_workflow_agent` — Simple Checks & Testing

This notebook performs incremental sanity checks on the workflow agent at `src/lg_workflow_agent`:

1. Environment & imports
2. Tool checks (`fetch_trends`, URL validator)
3. Prompt / state shape checks
4. Classifier node in isolation
5. Task generator node in isolation
6. Aggregator node with stubbed sub-agent outputs
7. Validator node with broken-link rewrite path (no LLM)
8. Full end-to-end run (sync)
9. Streaming run with per-step trace

Run cells top-to-bottom. Cells 4-6 and 8-9 require `GOOGLE_API_KEY`.

In [2]:
# 1. Environment & imports
import os
import sys
from pathlib import Path

# Make `src` importable when running the notebook from tests/
BACKEND_ROOT = Path.cwd()
if BACKEND_ROOT.name == "tests":
    BACKEND_ROOT = BACKEND_ROOT.parent
if str(BACKEND_ROOT) not in sys.path:
    sys.path.insert(0, str(BACKEND_ROOT))

from dotenv import load_dotenv
load_dotenv(BACKEND_ROOT / ".env")

print("BACKEND_ROOT:", BACKEND_ROOT)
print("GOOGLE_API_KEY set:", bool(os.environ.get("GOOGLE_API_KEY")))
print("DEEP_AGENT_MODEL:", os.environ.get("DEEP_AGENT_MODEL", "gemini-2.5-flash (default)"))
print("QDRANT_URL set:", bool(os.environ.get("QDRANT_URL")))

BACKEND_ROOT: /workspaces/aion-ai-research/backend
GOOGLE_API_KEY set: True
DEEP_AGENT_MODEL: google_genai:gemini-2.5-flash
QDRANT_URL set: True


## 2. Tool checks

`fetch_trends` (HTTP) and the URL validator (`validate_url`) used by the Validator node.

In [3]:
import time
from src.lg_workflow_agent.tools import (
    fetch_arxiv,
    fetch_github,
    fetch_google_news,
    fetch_hackernews,
    fetch_linkedin,
    fetch_podcasts,
    fetch_reddit,
    fetch_rss,
    fetch_youtube,
    validate_url,
    validate_urls,
    extract_urls,
    think_tool
)


In [4]:
import inspect
import json

async def test_tool(tool, topic: str = "langgraph", limit: int = 3, period: str = "week"):
    """Invoke a LangChain @tool with only the kwargs its underlying function accepts."""
    fn = getattr(tool, "func", None) or getattr(tool, "coroutine", None) or tool
    try:
        sig = inspect.signature(fn)
        params = set(sig.parameters)
    except (TypeError, ValueError):
        params = {"topic", "limit", "period"}

    kwargs = {"topic": topic, "limit": limit}
    if "period" in params:
        kwargs["period"] = period

    try:
        start_time = time.time()
        result = await tool.ainvoke(kwargs)
        end_time = time.time()
        preview = (result or "")[:300]
        print(f"[OK]   {tool.name:18s} ->  (took {end_time - start_time:.2f}s)\n")
        print(f"Full result_{tool.name} :\n{json.dumps(result, indent=2)}\n")
        return result
    except Exception as e:
        end_time = time.time()
        print(f"[FAIL] {tool.name:18s} -> {type(e).__name__}: {e} (took {end_time - start_time:.2f}s)\n")
        return None


In [5]:
# Run every fetch_* tool against the same topic and preview results
TOPIC = "langgraph"

fetch_tools = [
    fetch_arxiv,
    fetch_github,
    fetch_google_news,
    fetch_hackernews,
    fetch_linkedin,
    fetch_podcasts,
    fetch_reddit,
    fetch_rss,
    fetch_youtube,
]

for t in fetch_tools:
    await test_tool(t, topic=TOPIC, limit=3, period="week")


[OK]   fetch_arxiv        ->  (took 0.94s)

Full result_fetch_arxiv :
"{\"results\":[{\"title\":\"Compiling Agentic Workflows into LLM Weights: Near-Frontier Quality at Two Orders of Magnitude Less Cost\",\"url\":\"http://arxiv.org/abs/2605.22502v1\",\"source\":\"arxiv\",\"metadata\":{\"authors\":[\"Simon Dennis\",\"Rivaan Patil\",\"Kevin Shabahang\",\"Hao Guo\"],\"author_count\":4,\"abstract\":\"Agent orchestration frameworks have proliferated, collectively exceeding 290,000 GitHub stars across LangGraph, CrewAI, Google ADK, OpenAI Agents SDK, Semantic Kernel, Strands, and LlamaIndex. All follow the same pattern: an external orchestrator above the LLM, injecting instructions and routing decisions every turn. Recent work has shown this architecture is dominated for procedural tasks by simply providing the procedure in a frontier model's system prompt [Dennis et al., 2026a], at the cost o\",\"categories\":[\"cs.AI\",\"cs.LG\"],\"pdf_url\":\"http://arxiv.org/pdf/2605.22502v1\",\"publishe

In [6]:
# URL helpers (plain functions) + think_tool (@tool)
text = "See https://www.python.org and https://this-domain-should-not-exist-xyz123.invalid for refs."
urls = extract_urls(text)
print("extracted:", urls)
print("validate_url(python.org):", validate_url("https://www.python.org"))
print("validate_urls:", validate_urls(urls))

print("think_tool:", think_tool.invoke({"reflection": "Testing tool wiring end-to-end."}))


extracted: ['https://www.python.org', 'https://this-domain-should-not-exist-xyz123.invalid']
validate_url(python.org): True
validate_urls: {'https://www.python.org': True, 'https://this-domain-should-not-exist-xyz123.invalid': False}
think_tool: Reflection recorded: Testing tool wiring end-to-end.


## 3. Prompt / state shape checks

Quick sanity that prompts format correctly and the state TypedDict carries expected keys.

In [7]:
from src.lg_workflow_agent import prompts as P
from src.lg_workflow_agent.state import WorkflowState
# from src.lg_workflow_agent.nodes import ROLES_BY_TYPE, ROLE_PROMPTS
from src.lg_workflow_agent.nodes._constants import ROLE_PROMPTS,ROLES_BY_TYPE

# Required state keys
required_keys = {
    "query", "task_id", "query_type", "subtasks", "worker_payloads",
    "worker_outputs", "aggregated", "draft_report", "final_report",
    "validation_feedback", "invalid_references", "rewrite_iterations",
}
missing = required_keys - set(WorkflowState.__annotations__.keys())
assert not missing, f"State missing keys: {missing}"
print("State keys OK")

# Roles per type
for qt in ["blog", "comparative", "deep_research", "summary"]:
    roles = ROLES_BY_TYPE[qt]
    for r in roles:
        assert r in ROLE_PROMPTS, f"missing prompt for role {r}"
    print(f"  {qt:14s} -> {roles}")

# Prompt formatting smoke
print("\nPrompt formatting checks:")
print(P.CLASSIFIER_PROMPT.format(query="hello"))

print(f'\n ---------------------\n TASK_GENERATOR_PROMPT:')
print(P.TASK_GENERATOR_PROMPT.format(query="x", query_type="blog", roles="- web_research"))

State keys OK
  blog           -> ['web_research', 'latest_news_collection']
  comparative    -> ['web_research', 'latest_news_collection']
  deep_research  -> ['data_collection', 'statistics', 'citation']
  summary        -> ['web_research', 'latest_news_collection']

Prompt formatting checks:
You are the Query Classifier for a research-content workflow.

Step 1 — Mark the query as "ambiguous" if ANY of these are true:
- Too vague or generic (e.g. "tell me something", "research stuff").
- Unclear, incoherent, or missing context (e.g. "compare them").
- Mixes unrelated topics (e.g. "quantum computing vs pizza recipes").
- Just a few keywords with no clear research intent
  (e.g. "data analysis real estate", "AI healthcare").
- A short factual / yes-no / technical Q&A that does not need a researched
  article (e.g. "is training data dependent on number of labels?",
  "what is the capital of France?", "how do I install numpy?").
- Non-research input: greetings, chit-chat, opinions, jailb

In [8]:
from typing import TypedDict
from langgraph import graph
from langgraph.graph import START, END


builder = graph.StateGraph(WorkflowState)
# builder.add_node("passthrough", passthrough_node)
# builder.add_edge(START, "passthrough")
# builder.add_edge("passthrough", END)

# simple_graph = builder.compile()

# result = simple_graph.invoke({"message": "hello"})
# print(result)

In [ ]:
# create langrgraph workflow 
import langgraph.graph 

dict_keys(['query', 'task_id', 'messages', 'query_type', 'classification_rationale', 'ambiguous_reason', 'is_ambiguous', 'subtasks', 'worker_payloads', 'worker_outputs', 'aggregated', 'draft_report', 'final_report', 'chart_specs', 'report_images', 'validation_feedback', 'invalid_references', 'rewrite_iterations', 'research_paper_latex', 'research_paper_metadata', 'research_paper_pdf_base64', 'research_paper_images'])

## 4. Classifier node (isolated)

Initialize an LLM and run only the classifier on a few sample queries.

In [9]:
from langchain_google_genai import ChatGoogleGenerativeAI
from src.lg_workflow_agent.nodes import (
    create_node_classifier,
    create_node_task_generator,
    create_node_aggregator,
    create_node_validator,
    create_role_nodes
)

model_name = os.environ.get("DEEP_AGENT_MODEL", "gemini-2.5-flash")
model_name = "gemini-2.5-flash-lite"
if model_name.startswith("google_genai:"):
    model_name = model_name.split(":", 1)[1]

llm = ChatGoogleGenerativeAI(model=model_name, temperature=0.0)

# db=None: skip persistence to focus on logic
classify = create_node_classifier(llm, db=None)

samples = [
    "Compare LangGraph vs CrewAI vs AutoGen for multi-agent orchestration",
    "Write a blog post about async Python in 2026",
    "Give me a quick summary of vector databases",
    "Conduct an investigation on quantization-aware training benchmarks",
    "how to sample the data for classification model ",
    "is numpy good?"

]
for q in samples:
    start_time = time.time()
    out = classify({"query": q, "task_id": "demo"})
    end_time = time.time()
    print(f"{q[:55]:55s} -> {out['query_type']:14s} |Time : {end_time - start_time:.2f}s | {out['classification_rationale']}")

Compare LangGraph vs CrewAI vs AutoGen for multi-agent  -> comparative    |Time : 0.93s | The user explicitly asks to compare three distinct tools for multi-agent orchestration.
Write a blog post about async Python in 2026            -> blog           |Time : 0.99s | The user wants an explanatory article about a specific topic (async Python) with a future outlook.
Give me a quick summary of vector databases             -> summary        |Time : 0.86s | The user is requesting a concise overview of vector databases.
Conduct an investigation on quantization-aware training -> deep_research  |Time : 0.85s | The query requests a rigorous investigation into quantization-aware training benchmarks, implying a need for in-depth analysis and potentially extensive data.
how to sample the data for classification model         -> blog           |Time : 3.47s | The user is asking for an explanation of a specific technique for machine learning model development.
is numpy good?                         

In [10]:
builder = graph.StateGraph(WorkflowState)
builder.add_node("classify", classify)
builder.add_edge(START, "classify")
builder.add_edge("classify", END)
simple_graph = builder.compile()
start_time = time.time()
result = simple_graph.invoke({"query": "What is LangGraph?", "task_id": "demo"})
end_time = time.time()
print(f"Execution time: {end_time - start_time} seconds")
result

Execution time: 0.9521760940551758 seconds


{'query': 'What is LangGraph?',
 'task_id': 'demo',
 'messages': [],
 'query_type': 'blog',
 'classification_rationale': 'The user is asking for an explanation of a specific topic, LangGraph, which is suitable for an explanatory blog post.',
 'ambiguous_reason': '',
 'is_ambiguous': False,
 'worker_outputs': []}

## 5. Task generator node (isolated)

Verify the task generator decomposes a query and assigns roles allowed by the query type.

In [11]:
task_gen = create_node_task_generator(llm, db=None)

state = {
    "query": "Compare LangGraph vs CrewAI for multi-agent orchestration",
    "query_type": "comparative",
    "task_id": "demo",
}
result = task_gen(state)

print(f"Generated {len(result['subtasks'])} subtasks:")
for st in result["subtasks"]:
    print(f"  [{st['id']}] role={st['role']:18s} task={st['task']}")
print("\nWorker payloads (first):")
print(result["worker_payloads"][0])

# Assert role constraints
allowed = set(ROLES_BY_TYPE["comparative"])
assert all(st["role"] in allowed for st in result["subtasks"]), "Role outside allowed set"
print("\nRole assignment OK")

Generated 3 subtasks:
  [s1] role=web_research       task=Gather information on LangGraph's architecture, key features, and use cases for multi-agent orchestration, including any official documentation or reputable blog posts.
  [s2] role=web_research       task=Gather information on CrewAI's architecture, key features, and use cases for multi-agent orchestration, including any official documentation or reputable blog posts.
  [s3] role=web_research       task=Find and summarize comparisons or benchmarks between LangGraph and CrewAI for multi-agent orchestration from technical blogs, forums, or developer communities.

Worker payloads (first):
{'task_id': 'demo', 'query': 'Compare LangGraph vs CrewAI for multi-agent orchestration', 'subtask_id': 's1', 'role': 'web_research', 'task': "Gather information on LangGraph's architecture, key features, and use cases for multi-agent orchestration, including any official documentation or reputable blog posts."}

Role assignment OK


In [17]:
builder = graph.StateGraph(WorkflowState)
builder.add_node("classify", classify)
builder.add_node("task_gen", task_gen)
builder.add_edge(START, "classify")
builder.add_edge("classify", "task_gen")
builder.add_edge("task_gen", END)
lg_graph = builder.compile()


start_time = time.time()
result1 = lg_graph.invoke({"query": "Compare LangGraph vs CrewAI for multi-agent orchestration", "task_id": "demo"})
end_time = time.time()
print(f"Execution time: {end_time - start_time} seconds")
result1

Execution time: 3.913444757461548 seconds


{'query': 'Compare LangGraph vs CrewAI for multi-agent orchestration',
 'task_id': 'demo',
 'messages': [],
 'query_type': 'comparative',
 'classification_rationale': 'The user wants a comparison between two specific tools for multi-agent orchestration.',
 'ambiguous_reason': '',
 'is_ambiguous': False,
 'subtasks': [{'id': 's1',
   'role': 'web_research',
   'task': "Gather information on LangGraph's architecture, key features, and use cases for multi-agent orchestration, including any official documentation or reputable blog posts.",
   'status': 'pending'},
  {'id': 's2',
   'role': 'web_research',
   'task': "Gather information on CrewAI's architecture, key features, and use cases for multi-agent orchestration, including any official documentation or reputable blog posts.",
   'status': 'pending'},
  {'id': 's3',
   'role': 'web_research',
   'task': 'Find and summarize comparisons or benchmarks between LangGraph and CrewAI for multi-agent orchestration from technical blogs, forums

In [13]:
# 5b. Build sub-agent runners (one per role)
# `create_role_nodes` constructs each sub-agent once via `build_role_runners`
# and returns {node_name: async_runner}.
sub_agent_dict = create_role_nodes(llm)
print("Available sub-agent nodes:")
for name in sub_agent_dict.keys():
    print(f"  - {name}")

Available sub-agent nodes:
  - data_collection_agent
  - statistics_agent
  - citation_agent
  - web_research_agent
  - latest_news_collection_agent


In [14]:
# Invoke a single sub-agent directly (async runner).
# Payload mirrors the `worker_payloads` entries built by task_generator.
payload = {
    "task_id": "demo",
    "query": "Compare LangGraph vs CrewAI for multi-agent orchestration",
    "subtask_id": "s1",
    "role": "data_collection",
    "task": "Collect recent posts and discussions comparing LangGraph and CrewAI.",
}

start_time = time.time()
op = await sub_agent_dict["data_collection_agent"](payload)
print(f"Took {time.time() - start_time:.1f}s")
print("Keys:", list(op.keys()))
print(json.dumps(op, indent=2))

Took 6.9s
Keys: ['worker_outputs']
{
  "worker_outputs": [
    {
      "subtask_id": "s1",
      "role": "data_collection",
      "task": "Collect recent posts and discussions comparing LangGraph and CrewAI.",
      "output": "I was unable to find any recent posts or discussions directly comparing LangGraph and CrewAI on Reddit, Google News, or Hacker News. My attempts to search Reddit were blocked, and the other platforms returned no relevant results. This suggests that direct comparisons may not be widely available or trending at the moment.\n\nTo gather information, I will need to broaden my search to include general articles on multi-agent systems and then look for mentions of LangGraph and CrewAI within those. Alternatively, I could investigate the documentation and use cases for each framework separately to infer comparisons."
    }
  ]
}


In [ ]:
result1['subtasks']

[{'id': 's1',
  'role': 'web_research',
  'task': "Gather information on LangGraph's architecture, key features, and use cases for multi-agent orchestration, including any official documentation or reputable blog posts.",
  'status': 'pending'},
 {'id': 's2',
  'role': 'web_research',
  'task': "Gather information on CrewAI's architecture, key features, and use cases for multi-agent orchestration, including any official documentation or reputable blog posts.",
  'status': 'pending'},
 {'id': 's3',
  'role': 'web_research',
  'task': 'Find and summarize comparisons or benchmarks between LangGraph and CrewAI for multi-agent orchestration from technical blogs, forums, or developer communities.',
  'status': 'pending'}]

In [ ]:
# Test EVERY sub-agent runner against the same query.
# Each runner returns {"worker_outputs": [{subtask_id, role, task, output}]}.
ROLE_TASKS = {
    "data_collection": "Collect recent community posts comparing LangGraph and CrewAI.",
    "statistics": "Find quantitative benchmarks or adoption metrics for LangGraph vs CrewAI.",
    "citation": "Locate authoritative references (papers/docs) for LangGraph and CrewAI.",
    "web_research": "Summarize how LangGraph and CrewAI differ for multi-agent orchestration.",
    "latest_news_collection": "Gather the latest news/releases about LangGraph and CrewAI.",
}

QUERY = "Compare LangGraph vs CrewAI for multi-agent orchestration"
sub_agent_results = {}1

for i, (role, task) in enumerate(ROLE_TASKS.items(), start=1):
    node_name = f"{role}_agent"
    runner = sub_agent_dict.get(node_name)
    if runner is None:
        print(f"[SKIP] {node_name} (not registered)")
        continue
    payload = {
        "task_id": "demo",
        "query": QUERY,
        "subtask_id": f"s{i}",
        "role": role,
        "task": task,
    }
    t0 = time.time()
    try:
        result = await runner(payload)
        out = result["worker_outputs"][0]
        preview = (out["output"] or "")[:240].replace("\n", " ")
        print(f"[OK]   {node_name:30s} {time.time()-t0:5.1f}s  chars={len(out['output'])}")
        print(f"       preview: {preview}\n")
        sub_agent_results[node_name] = result
    except Exception as e:
        print(f"[FAIL] {node_name:30s} {time.time()-t0:5.1f}s  {type(e).__name__}: {e}\n")

[OK]   data_collection_agent           13.1s  chars=19
       preview: No output produced.

[OK]   statistics_agent                 6.3s  chars=1027
       preview: ## Key Statistics - **GitHub Stars (LangGraph related repos):** Up to 69,379 stars for `bytedance/deer-flow` and 23,264 for `langchain-ai/deepagents`. (May 2026) [1] - **GitHub Stars (CrewAI):** 52,076 stars for the main `crewAIInc/crewAI` 

[OK]   citation_agent                  10.6s  chars=1859
       preview: ## References [1] Compiling Agentic Workflows into LLM Weights: Near-Frontier Quality at Two Orders of Magnitude Less Cost - http://arxiv.org/abs/2605.22502v1 - Discusses agent orchestration frameworks including LangGraph and CrewAI. [2] En

[OK]   web_research_agent              22.8s  chars=19
       preview: No output produced.

[OK]   latest_news_collection_agent     6.5s  chars=519
       preview: It appears that there is limited direct news or recent releases specifically comparing LangGraph and CrewAI. My se

In [16]:
out

{'subtask_id': 's5',
 'role': 'latest_news_collection',
 'task': 'Gather the latest news/releases about LangGraph and CrewAI.',
 'output': 'It appears that there is limited direct news or recent releases specifically comparing LangGraph and CrewAI. My searches on Google News, Hacker News, and RSS feeds did not yield direct comparisons or recent updates on this topic. Reddit searches were also blocked.\n\nTo provide a comparison, I would need to gather more information, potentially by looking for general news about each tool and then synthesizing a comparison, or by searching for more technical discussions or tutorials that might implicitly compare them.'}

In [1]:
# Build a minimal LangGraph wiring a sub-agent into a graph.
# classify -> task_gen -> assign_workers -> (Send to sub-agents) -> END
from src.lg_workflow_agent.nodes import create_assign_workers

assign_workers = create_assign_workers()

builder = graph.StateGraph(WorkflowState)
builder.add_node("classify", classify)
builder.add_node("task_gen", task_gen)
for node_name, runner in sub_agent_dict.items():
    builder.add_node(node_name, runner)

builder.add_edge(START, "classify")
builder.add_edge("classify", "task_gen")
# assign_workers returns a list of Send(...) objects -> conditional fan-out edge
builder.add_conditional_edges(
    "task_gen",
    assign_workers,
    list(sub_agent_dict.keys()),
)
for node_name in sub_agent_dict.keys():
    builder.add_edge(node_name, END)

sub_agent_graph = builder.compile()
print("Graph compiled with nodes:", list(sub_agent_dict.keys()))


NameError: name 'graph' is not defined

In [ ]:
# Run the sub-agent graph end-to-end and inspect worker_outputs.
start_time = time.time()
result = await sub_agent_graph.ainvoke(
    {"query": "Compare LangGraph vs CrewAI for multi-agent orchestration", "task_id": "demo"}
)
print(f"Execution time: {time.time() - start_time:.1f}s")
print(f"query_type : {result.get('query_type')}")
print(f"subtasks   : {len(result.get('subtasks', []))}")
print(f"outputs    : {len(result.get('worker_outputs', []))}\n")

for wo in result.get("worker_outputs", []):
    preview = (wo.get("output") or "")[:200].replace("\n", " ")
    print(f"[{wo.get('subtask_id')}] {wo.get('role'):22s} -> {preview}")


## 6. Aggregator with stubbed sub-agent outputs

Skip real sub-agent calls and feed pre-baked outputs to confirm the aggregator
returns a structured `{metadata, sections, references}` object.

In [ ]:
import json

aggregate = create_node_aggregator(llm, db=None)

stub_outputs = [
    {
        "subtask_id": "s1",
        "role": "web_research",
        "task": "Research LangGraph orchestration model",
        "output": (
            "## Findings\n"
            "LangGraph models workflows as a stateful graph with explicit nodes and edges [1].\n"
            "It supports streaming and human-in-the-loop checkpoints [2].\n"
            "## Sources\n"
            "[1] LangGraph Docs - https://langchain-ai.github.io/langgraph/\n"
            "[2] LangGraph blog - https://blog.langchain.dev/langgraph/"
        ),
    },
    {
        "subtask_id": "s2",
        "role": "content_drafting",
        "task": "Research CrewAI orchestration model",
        "output": (
            "## Draft\n"
            "CrewAI uses a role-based crew abstraction with sequential or hierarchical processes [1].\n"
            "## Sources\n"
            "[1] CrewAI Docs - https://docs.crewai.com/"
        ),
    },
]

state = {
    "query": "Compare LangGraph vs CrewAI",
    "query_type": "comparative",
    "task_id": "demo",
    "worker_outputs": stub_outputs,
}
agg = aggregate(state)["aggregated"]
print(json.dumps(agg, indent=2)[:1500])

## 7. Validator (no LLM)

The validator extracts URLs from the draft, verifies them, and signals a rewrite when broken refs are found.

In [ ]:
validator = create_node_validator(db=None)

draft_good = "Read more at https://www.python.org for details."
draft_bad = (
    "Reference [1] points to https://www.python.org but [2] is broken: "
    "https://this-domain-should-not-exist-xyz123.invalid"
)

print("good draft:")
print(validator({"draft_report": draft_good, "rewrite_iterations": 0}))

print("\nbad draft (1st pass -> rewrite):")
print(validator({"draft_report": draft_bad, "rewrite_iterations": 0}))

print("\nbad draft (after MAX_REWRITES -> forced finish):")
print(validator({"draft_report": draft_bad, "rewrite_iterations": 2}))

## 8. End-to-end (sync)

Build a fresh `WorkflowAgent` and run a single query through the full graph.

In [ ]:
from src.lg_workflow_agent import WorkflowAgent

agent = WorkflowAgent()
agent.build()
print("Agent ready:", agent.is_ready)

QUERY = "Give a short summary of LangGraph for multi-agent orchestration"
report = agent.invoke(QUERY)
print("\n--- FINAL REPORT ---\n")
print(report)

## 9. Streaming run with per-step trace

Use `astream` to observe how state evolves across nodes (classifier, task_generator, sub-agents, aggregator, writer, validator, cleanup).

In [ ]:
import json

STREAM_QUERY = "Conduct a deep research investigation on quantization-aware training benchmarks"

trace = []
final_report = ""
event_count = 0
async for event in agent.astream(STREAM_QUERY):
    step = event.get("step")
    data = event.get("data", {})
    print(f"-----------------------\nEvent: step={step} |\n  data : ={json.dumps(data, indent=2)} \n\n~~~~~~~~~~~~~~~~~~~~~~~~~~")
    # keys = sorted(k for k in data.keys() if k != "messages")
    # line = f"[{step}] keys={keys}"
    # if data.get("query_type"):
    #     line += f" | query_type={data['query_type']}"
    # if data.get("subtasks"):
    #     line += f" | subtasks={len(data['subtasks'])}"
    # if data.get("validation_feedback"):
    #     line += f" | validation={data['validation_feedback']}"
    # if data.get("web_research_agent"):
    #     line += f" | web_research_agent={data['web_research_agent']}"
    # print(line)
    # trace.append(line)
    if data.get("final_report"):
        final_report = data["final_report"]
    elif data.get("draft_report"):
        final_report = data["draft_report"]
    event_count += 1

print(f"\nTotal events: {event_count}")
print("\n--- FINAL REPORT ---\n")
print(final_report or "(no report)")

In [ ]:
import json

STREAM_QUERY = "what is latest work in data science"

trace = []
final_report = ""
event_count = 0
async for event in agent.astream(STREAM_QUERY):
    step = event.get("step")
    data = event.get("data", {})
    print(f"-----------------------\nEvent: step={step} |\n  data : ={json.dumps(data, indent=2)} \n\n~~~~~~~~~~~~~~~~~~~~~~~~~~")
    # keys = sorted(k for k in data.keys() if k != "messages")
    # line = f"[{step}] keys={keys}"
    # if data.get("query_type"):
    #     line += f" | query_type={data['query_type']}"
    # if data.get("subtasks"):
    #     line += f" | subtasks={len(data['subtasks'])}"
    # if data.get("validation_feedback"):
    #     line += f" | validation={data['validation_feedback']}"
    # if data.get("web_research_agent"):
    #     line += f" | web_research_agent={data['web_research_agent']}"
    # print(line)
    # trace.append(line)
    if data.get("final_report"):
        final_report = data["final_report"]
    elif data.get("draft_report"):
        final_report = data["draft_report"]
    event_count += 1

print(f"\nTotal events: {event_count}")
print("\n--- FINAL REPORT ---\n")
print(final_report or "(no report)")